# end-grad-default-ones-like — worked example 1: ones_like Matches Shape and dtype of the End Node

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `end-grad-default-ones-like`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you call `loss.backward()` without arguments, PyTorch implicitly seeds the reverse pass with a gradient of `1.0` — the identity for chain-rule multiplication. The function `torch.ones_like(end_node)` creates this seed tensor with the same shape, dtype, and device as the end node. For a scalar loss this is `tensor(1.0)`, but for a per-sample loss vector of shape `(B,)` it is `ones(B)`, correctly seeding independent backpropagation for each sample.

## Worked solution

We demonstrate `ones_like` on three different end-node shapes: a scalar, a 1-D vector, and a 2-D matrix.

**Scalar case:** `end_node.shape = ()`. `torch.ones_like(end_node)` returns `tensor(1.)` with shape `()`. This is the standard `.backward()` seed for scalar losses.

**Vector case:** `end_node.shape = (4,)`. `torch.ones_like(end_node)` returns `tensor([1., 1., 1., 1.])` with shape `(4,)`. This seeds independent gradients for each element of a per-sample loss vector.

**2-D case:** `end_node.shape = (3, 3)`. `torch.ones_like(end_node)` returns a 3×3 tensor of ones. Same dtype and device as the original.

The key property in every case: the seed has exactly the same shape and dtype as the end node, and all values are 1 (the multiplicative identity for chain-rule products).

In [ ]:
import torch as t

# Demonstrate ones_like on three end-node shapes

def default_end_grad(end_array):
    """Return ones_like(end_array) — the default backward seed."""
    return t.ones_like(end_array)

# Case 1: scalar
scalar_node = t.tensor(3.7)
seed_scalar = default_end_grad(scalar_node)
print(f"Scalar end node shape: {scalar_node.shape}, seed shape: {seed_scalar.shape}, value: {seed_scalar}")
assert seed_scalar.shape == scalar_node.shape
assert seed_scalar.dtype == scalar_node.dtype
assert seed_scalar.item() == 1.0

# Case 2: 1-D vector (per-sample losses)
vec_node = t.tensor([0.5, 1.2, 0.8, 2.1])
seed_vec = default_end_grad(vec_node)
print(f"Vector end node shape: {vec_node.shape}, seed shape: {seed_vec.shape}")
assert seed_vec.shape == (4,)
assert (seed_vec == 1.0).all()

# Case 3: 2-D matrix
mat_node = t.randn(3, 3)
seed_mat = default_end_grad(mat_node)
print(f"Matrix end node shape: {mat_node.shape}, seed shape: {seed_mat.shape}")
assert seed_mat.shape == (3, 3)
assert seed_mat.dtype == mat_node.dtype
assert (seed_mat == 1.0).all()

print("All cases: ones_like produces correct shape and dtype.")